# EV Grid Oracle — GRPO Training (Colab T4)

This notebook trains a small LLM (Qwen 2.5 3B Instruct) with **verifier-based GRPO** to route EVs in the `EVGridCore` simulation.

**Action schema (strict):**

```
ACTION: route|defer|load_shift
STATION: BLR-01..BLR-25 or NONE
CHARGE_RATE: slow|fast|ultra_fast
DEFER_MINUTES: integer
REASON: max 20 words
CONFIDENCE: 0.0-1.0
```

> If you’re using LoRA/QLoRA, don’t naively upcast a 4-bit base to 16-bit and “merge” at the end without the correct path — it can badly degrade quality. Save adapters cleanly and test post-training inference immediately.


In [ ]:
# Install (Colab)
!pip install -q unsloth trl transformers accelerate datasets
!pip install -q pydantic networkx openenv-core fastapi uvicorn pytest

# If you're running from a cloned repo, install it too:
# !pip install -e .


In [ ]:
import re
from typing import Optional

from datasets import Dataset

from ev_grid_oracle.city_graph import build_city_graph
from ev_grid_oracle.env import EVGridCore
from ev_grid_oracle.models import ActionType, ChargeRate, EVGridAction, GridState


graph = build_city_graph()
core = EVGridCore(city_graph=graph)


In [ ]:
ACTION_RE = re.compile(
    r"ACTION:\s*(?P<action>route|defer|load_shift)\s*\n"
    r"STATION:\s*(?P<station>BLR-\d\d|NONE)\s*\n"
    r"CHARGE_RATE:\s*(?P<rate>slow|fast|ultra_fast)\s*\n"
    r"DEFER_MINUTES:\s*(?P<defer>\d+)\s*\n",
    re.IGNORECASE,
)


def parse_action(text: str, *, ev_id: str) -> Optional[EVGridAction]:
    m = ACTION_RE.search(text.strip())
    if not m:
        return None

    action_type = ActionType(m.group("action").lower())
    station = m.group("station").upper()
    rate = ChargeRate(m.group("rate").lower())
    defer = int(m.group("defer"))

    station_id = None if station == "NONE" else station

    try:
        return EVGridAction(
            action_type=action_type,
            ev_id=ev_id,
            station_id=station_id,
            charge_rate=rate,
            defer_minutes=defer,
        )
    except Exception:
        return None


In [ ]:
def generate_episode_dataset(n: int = 500, *, seed: int = 123) -> Dataset:
    rows = []
    for i in range(n):
        obs = core.reset(seed=seed + i)
        # Keep verifier honest: reward_fn uses state_json, not prompt parsing.
        rows.append(
            {
                "prompt": obs.prompt,
                "state_json": obs.state.model_dump(mode="json"),
            }
        )
    return Dataset.from_list(rows)


train_ds = generate_episode_dataset(n=500)
train_ds[0]["prompt"][:400]


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)


In [ ]:
def reward_fn(prompts, responses, **kwargs):
    rewards = []

    # TRL passes prompt strings; we recover the matching state via dataset column.
    # We rely on GRPOTrainer passing `kwargs["batch"]` with original examples.
    batch = kwargs.get("batch")
    state_jsons = batch["state_json"] if batch is not None and "state_json" in batch else None

    for prompt, response, state_json in zip(prompts, responses, state_jsons or [None] * len(prompts)):
        if state_json is None:
            rewards.append(0.0)
            continue

        # pick target ev_id = first pending EV in state (matches prompt_builder v0)
        state = GridState.model_validate(state_json)
        ev_id = state.pending_evs[0].ev_id if state.pending_evs else "EV-000"

        action = parse_action(response, ev_id=ev_id)
        if action is None:
            rewards.append(0.0)
            continue

        local = EVGridCore(city_graph=graph)
        local._grid_state = state
        _ = local.step(action)
        rewards.append(float(_.reward_breakdown.get("total", 0.0)))

    return rewards


In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir="ev_oracle_grpo",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    num_generations=4,
    max_completion_length=160,
    report_to=[],
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=config,
    train_dataset=train_ds,
)

trainer.train()


In [ ]:
# Save adapters (recommended) and a merged artifact (optional)
model.save_pretrained("ev_oracle_lora")
tokenizer.save_pretrained("ev_oracle_lora")

# Optional merged export (test immediately after):
# model.save_pretrained_merged("ev_oracle_merged_16bit", tokenizer, save_method="merged_16bit")
